# GenLab — Benchmark de Descarga: HuggingFace vs Google Drive

Benchmark completo de velocidad de transferencia.

**Mide:**
- Descarga desde HuggingFace (modos normal y xet_high)
- Subida a Google Drive
- Descarga desde Google Drive
- max_workers sweep (1→8) en HF

**Basado en:** HF_Download_Benchmark (ResourceMonitor, stalls, CPU/RAM)

**Requisitos:**
1. Runtime con GPU (T4 o superior)
2. Espacio libre suficiente (~8.2 GB para DreamShaper)
3. Google Drive montado

In [ ]:
# ---- 1. INSTALACION ----
import os
!pip uninstall -y hf-xet 2>/dev/null
!pip install -qU huggingface-hub==0.27.0 psutil pandas matplotlib tqdm pyyaml
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'

In [ ]:
# ---- 2. MONTAR GOOGLE DRIVE ----
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BENCH_DIR = '/content/drive/MyDrive/genlab_benchmark'
!mkdir -p "$DRIVE_BENCH_DIR"

In [ ]:
# ---- 3. CLONAR REPO + IMPORTS ----
import os, sys, time, threading, json, shutil, warnings, math
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

import psutil
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('retina')

from huggingface_hub import snapshot_download, HfApi
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

REPO_URL = 'https://github.com/ke1npro/GenLab.git'
GENLAB_SRC = '/content/genlab'

if os.path.isdir(GENLAB_SRC):
    !git -C $GENLAB_SRC pull
else:
    !git clone $REPO_URL $GENLAB_SRC

sys.path.insert(0, f'{GENLAB_SRC}/src')
os.chdir(GENLAB_SRC)

from genlab import GenLab
from genlab.models.registry import detect_model
from genlab.config.loader import load_config

print('Ok')

## Detección de Entorno

In [ ]:
# ---- 4. DETECCION DE ENTORNO ----

def detect_environment():
    info = {}
    info['cpu_logical'] = psutil.cpu_count(logical=True)
    info['cpu_physical'] = psutil.cpu_count(logical=False)
    mem = psutil.virtual_memory()
    info['ram_total_gb'] = round(mem.total / 1024**3, 1)
    info['ram_available_gb'] = round(mem.available / 1024**3, 1)
    du = shutil.disk_usage('/')
    info['disk_total_gb'] = round(du.total / 1024**3, 1)
    info['disk_free_gb'] = round(du.free / 1024**3, 1)
    try:
        import subprocess
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                           '--format=csv,noheader'], capture_output=True, text=True, timeout=5)
        info['gpu'] = r.stdout.strip() if r.returncode == 0 else 'N/A'
    except:
        info['gpu'] = 'N/A'
    try:
        from huggingface_hub import whoami
        user = whoami()
        info['hf_user'] = user.get('name', user.get('email', 'logged in'))
    except:
        info['hf_user'] = 'NOT LOGGED IN'
    try:
        import requests
        r = requests.get('https://ipinfo.io/json', timeout=5)
        if r.ok:
            d = r.json()
            info['region'] = f"{d.get('city','?')}, {d.get('region','?')}, {d.get('country','?')}"
    except:
        info['region'] = 'N/A'
    return info

env = detect_environment()
for k, v in env.items():
    print(f'  {k:20s} -> {v}')

## Elegir Modelo

In [ ]:
# ---- 5. ELEGIR MODELO ----
DEFAULT_MODEL = "Lykon/dreamshaper-7"
DEFAULT_SIZE_GB = 8

print(f"Modelo por defecto: {DEFAULT_MODEL} (~{DEFAULT_SIZE_GB} GB)")
custom = input("Model ID custom (Enter para usar default): ").strip()
model_id = custom or DEFAULT_MODEL
print(f"\nModelo seleccionado: {model_id}")

## ResourceMonitor + Helpers

In [ ]:
# ---- 6. RESOURCE MONITOR + HELPERS ----

DOWNLOAD_BASE = Path('/content/hf_bench_downloads')

ESSENTIAL_ONLY = ['*.bin', '*.safetensors', '*.json', '*.txt', '*.md']


class ResourceMonitor:
    def __init__(self, download_dir: str, interval: float = 2.0):
        self.download_dir = Path(download_dir)
        self.interval = interval
        self._running = False
        self._thread: Optional[threading.Thread] = None
        self.records: list[dict] = []
        self.stall_events: list[dict] = []
        self._prev_write = psutil.disk_io_counters().write_bytes
        self._prev_size = self._get_dir_size()

    def _get_dir_size(self) -> int:
        total = 0
        for dirpath, _, filenames in os.walk(str(self.download_dir)):
            for f in filenames:
                try:
                    total += (Path(dirpath) / f).stat().st_size
                except OSError:
                    pass
        return total

    def start(self):
        self._running = True
        self._prev_size = self._get_dir_size()
        self._prev_write = psutil.disk_io_counters().write_bytes
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=10)

    def _run(self):
        low_speed_start = None
        while self._running:
            time.sleep(self.interval)
            try:
                curr_write = psutil.disk_io_counters().write_bytes
                curr_size = self._get_dir_size()
                io_write_mbps = (curr_write - self._prev_write) / (1024**2) / self.interval
                dl_speed = (curr_size - self._prev_size) / (1024**2) / self.interval
                record = {
                    'timestamp': time.time(),
                    'cpu_percent': psutil.cpu_percent(),
                    'ram_gb': round(psutil.virtual_memory().used / 1024**3, 2),
                    'ram_percent': psutil.virtual_memory().percent,
                    'disk_write_mbps': round(max(0, io_write_mbps), 1),
                    'download_speed_mbps': round(max(0, dl_speed), 1),
                    'downloaded_gb': round(curr_size / 1024**3, 3),
                }
                self.records.append(record)
                if 0 < dl_speed < 5:
                    if low_speed_start is None:
                        low_speed_start = time.time()
                    elif time.time() - low_speed_start > 10:
                        self.stall_events.append({
                            'time': str(datetime.now()),
                            'duration_s': round(time.time() - low_speed_start, 1),
                            'speed_mbps': round(dl_speed, 1),
                        })
                        low_speed_start = None
                else:
                    low_speed_start = None
                self._prev_write = curr_write
                self._prev_size = curr_size
            except Exception:
                pass

    def get_summary(self) -> dict:
        if not self.records:
            return {}
        speeds = [r['download_speed_mbps'] for r in self.records if r['download_speed_mbps'] > 0]
        return {
            'avg_download_speed_mbps': round(sum(speeds) / len(speeds), 2) if speeds else 0,
            'peak_download_speed_mbps': round(max(speeds), 2) if speeds else 0,
            'avg_cpu_percent': round(sum(r['cpu_percent'] for r in self.records) / len(self.records), 1),
            'peak_cpu_percent': max(r['cpu_percent'] for r in self.records),
            'avg_ram_percent': round(sum(r['ram_percent'] for r in self.records) / len(self.records), 1),
            'stall_count': len(self.stall_events),
            'stall_events': self.stall_events,
        }


def get_total_model_size(repo_id: str) -> int:
    try:
        api = HfApi()
        info = api.repo_info(repo_id, repo_type='model', files_metadata=True)
        return sum(f.size for f in info.siblings if f.size is not None)
    except Exception as e:
        print(f'  Warning: no se pudo obtener tamaño de {repo_id}: {e}')
        return 0


def clear_hf_cache():
    try:
        from huggingface_hub import scan_cache_dir
        cache_info = scan_cache_dir()
        if cache_info.size_on_disk > 0:
            strategy = cache_info.delete_revisions(*cache_info.revisions)
            strategy.execute()
    except Exception:
        cache_dir = Path(os.environ.get('HF_HOME', Path.home() / '.cache/huggingface'))
        if cache_dir.exists():
            shutil.rmtree(str(cache_dir), ignore_errors=True)
    if DOWNLOAD_BASE.exists():
        shutil.rmtree(str(DOWNLOAD_BASE), ignore_errors=True)
    DOWNLOAD_BASE.mkdir(parents=True, exist_ok=True)


def check_disk_space_needed(model_size_bytes: int) -> bool:
    free = shutil.disk_usage('/').free
    needed = model_size_bytes * 1.3
    if free < needed:
        print(f'  DISK SPACE LOW: free={free/1024**3:.1f}GB, needed={needed/1024**3:.1f}GB')
        return False
    return True


@dataclass
class BenchmarkResult:
    model_id: str
    method: str
    max_workers: int
    total_size_bytes: int
    total_time_s: float
    avg_speed_mbps: float
    peak_speed_mbps: float
    file_count: int
    avg_cpu_percent: float
    peak_cpu_percent: float
    avg_ram_percent: float
    stall_count: int
    stall_events: list = field(default_factory=list)
    error: Optional[str] = None


def run_download_benchmark(
    model_id: str,
    method: str = 'normal',
    max_workers: int = 4,
    allow_patterns: Optional[list] = None,
) -> BenchmarkResult:
    print(f'\n{"="*60}')
    print(f'  Model: {model_id}')
    print(f'  Config: method={method}, max_workers={max_workers}')
    if allow_patterns:
        print(f'  Patterns: {allow_patterns}')
    print('='*60)

    total_size = get_total_model_size(model_id)
    print(f'  Repo size: {total_size/1024**3:.1f} GB')

    if not check_disk_space_needed(total_size):
        return BenchmarkResult(
            model_id=model_id, method=method, max_workers=max_workers,
            total_size_bytes=total_size, total_time_s=0, avg_speed_mbps=0,
            peak_speed_mbps=0, file_count=0, avg_cpu_percent=0,
            peak_cpu_percent=0, avg_ram_percent=0, stall_count=0,
            error='Disk space insufficient'
        )

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    dest = DOWNLOAD_BASE / stamp
    dest.mkdir(parents=True)

    if method == 'xet_high':
        os.environ.pop('HF_HUB_DISABLE_XET', None)
        os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
    else:
        os.environ['HF_HUB_DISABLE_XET'] = '1'
        os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)

    monitor = ResourceMonitor(str(dest))
    monitor.start()

    t0 = time.perf_counter()
    error = None
    file_count = 0
    try:
        snapshot_download(
            repo_id=model_id,
            local_dir=str(dest),
            local_dir_use_symlinks=False,
            max_workers=max_workers,
            allow_patterns=allow_patterns,
            resume_download=True,
        )
        file_count = sum(1 for _ in Path(dest).rglob('*') if _.is_file())
    except Exception as e:
        error = str(e)[:200]
        print(f'  ERROR: {error}')
    t1 = time.perf_counter()

    monitor.stop()
    total_time = t1 - t0

    actual_size = sum(
        f.stat().st_size for f in Path(dest).rglob('*') if f.is_file()
    ) if not error else 0

    avg_speed = (actual_size / 1024**2) / total_time if total_time > 0 else 0
    mon_summary = monitor.get_summary()

    shutil.rmtree(str(dest), ignore_errors=True)

    return BenchmarkResult(
        model_id=model_id,
        method=method,
        max_workers=max_workers,
        total_size_bytes=actual_size,
        total_time_s=round(total_time, 1),
        avg_speed_mbps=round(avg_speed, 2),
        peak_speed_mbps=mon_summary.get('peak_download_speed_mbps', 0),
        file_count=file_count,
        avg_cpu_percent=mon_summary.get('avg_cpu_percent', 0),
        peak_cpu_percent=mon_summary.get('peak_cpu_percent', 0),
        avg_ram_percent=mon_summary.get('avg_ram_percent', 0),
        stall_count=mon_summary.get('stall_count', 0),
        stall_events=mon_summary.get('stall_events', []),
        error=error,
    )

## Benchmark 1: Descarga desde HuggingFace

In [ ]:
# ---- 7. BENCHMARK: HUGGINGFACE DOWNLOAD ----

hf_results: list[BenchmarkResult] = []

for method in ['normal', 'xet_high']:
    clear_hf_cache()
    r = run_download_benchmark(
        model_id,
        method=method,
        max_workers=6,
        allow_patterns=ESSENTIAL_ONLY,
    )
    hf_results.append(r)
    clear_hf_cache()

print('\nHF benchmark done')

## Benchmark 2: max_workers sweep

In [ ]:
# ---- 8. MAX_WORKERS SWEEP (HF download) ----

free_gb = shutil.disk_usage('/').free / 1024**3
model_size_gb = get_total_model_size(model_id) / 1024**3
max_possible_runs = int(free_gb / (model_size_gb * 1.3))

print(f'  Free disk: {free_gb:.1f} GB')
print(f'  Model size: {model_size_gb:.1f} GB')
print(f'  Possible runs: {max_possible_runs}')

if max_possible_runs >= 8:
    workers_to_test = [1, 2, 3, 4, 5, 6, 7, 8]
elif max_possible_runs >= 4:
    workers_to_test = [1, 2, 4, 8]
elif max_possible_runs >= 2:
    workers_to_test = [1, 8]
else:
    workers_to_test = [4]
    print('  Low disk space - single test only')

print(f'  Testing workers: {workers_to_test}')

mw_results: list[BenchmarkResult] = []
for w in workers_to_test:
    r = run_download_benchmark(
        model_id,
        method='normal',
        max_workers=w,
        allow_patterns=ESSENTIAL_ONLY,
    )
    mw_results.append(r)
    clear_hf_cache()

print('\nMax_workers benchmark done')

## Benchmark 3: Subida a Google Drive

In [ ]:
# ---- 9. SUBIDA A GOOGLE DRIVE ----

print('='*60)
print(f'  Drive Upload: {model_id}')
print('='*60)

DRIVE_MODEL_DIR = f"{DRIVE_BENCH_DIR}/{model_id.replace('/', '__')}"

# 1. Verificar espacio en Drive
drive_usage = shutil.disk_usage(DRIVE_BENCH_DIR)
print(f'  Drive free: {drive_usage.free/1024**3:.1f} GB')

model_size_bytes = get_total_model_size(model_id)
if drive_usage.free < model_size_bytes * 1.1:
    print(f'  ERROR: Drive space insufficient. Need {model_size_bytes/1024**3*1.1:.1f} GB, have {drive_usage.free/1024**3:.1f} GB')
    upload_result = {'error': 'Drive space insufficient'}
else:
    # 2. Verificar si ya existe
    if os.path.isdir(DRIVE_MODEL_DIR):
        existing_size = sum(f.stat().st_size for f in Path(DRIVE_MODEL_DIR).rglob('*') if f.is_file())
        print(f'  Ya existe en Drive: {existing_size/1024**3:.2f} GB')
        reuse = input('  Usar copia existente? (s/N): ').strip().lower()
        if reuse == 's':
            print('  Usando copia existente en Drive.')
            upload_result = {'skipped': True, 'path': DRIVE_MODEL_DIR}
        else:
            print('  Eliminando copia existente...')
            shutil.rmtree(DRIVE_MODEL_DIR)
    
    if not isinstance(upload_result, dict) or not upload_result.get('skipped'):
        # 3. Descargar a local (si no existe aún)
        DL_DIR = f'/content/models_cache/{model_id.replace("/", "__")}'
        if not os.path.isdir(DL_DIR) or not any(Path(DL_DIR).iterdir()):
            print(f'  Descargando {model_id} a caché local...')
            os.makedirs(DL_DIR, exist_ok=True)
            snapshot_download(repo_id=model_id, local_dir=DL_DIR, max_workers=6, allow_patterns=ESSENTIAL_ONLY)
        
        local_size = sum(f.stat().st_size for f in Path(DL_DIR).rglob('*') if f.is_file())
        print(f'  Cache local: {local_size/1024**3:.2f} GB')

        # 4. Subir a Drive
        print(f'  Subiendo a Drive...')
        monitor = ResourceMonitor(DRIVE_MODEL_DIR)
        monitor.start()
        t0 = time.perf_counter()
        try:
            shutil.copytree(DL_DIR, DRIVE_MODEL_DIR, symlinks=False)
            error = None
        except Exception as e:
            error = str(e)[:200]
            print(f'  ERROR: {error}')
        t1 = time.perf_counter()
        monitor.stop()

        total_time = t1 - t0
        actual_size = sum(f.stat().st_size for f in Path(DRIVE_MODEL_DIR).rglob('*') if f.is_file()) if not error else 0
        avg_speed = (actual_size / 1024**2) / total_time if total_time > 0 else 0
        mon_summary = monitor.get_summary()

        upload_result = BenchmarkResult(
            model_id=model_id,
            method='drive_upload',
            max_workers=0,
            total_size_bytes=actual_size,
            total_time_s=round(total_time, 1),
            avg_speed_mbps=round(avg_speed, 2),
            peak_speed_mbps=mon_summary.get('peak_download_speed_mbps', 0),
            file_count=sum(1 for _ in Path(DRIVE_MODEL_DIR).rglob('*') if _.is_file()) if not error else 0,
            avg_cpu_percent=mon_summary.get('avg_cpu_percent', 0),
            peak_cpu_percent=mon_summary.get('peak_cpu_percent', 0),
            avg_ram_percent=mon_summary.get('avg_ram_percent', 0),
            stall_count=mon_summary.get('stall_count', 0),
            stall_events=mon_summary.get('stall_events', []),
            error=error,
        )

upload_result_label = ' (usando existente)' if isinstance(upload_result, dict) and upload_result.get('skipped') else ''
print(f'  Upload done{upload_result_label}')

## Benchmark 4: Descarga desde Google Drive

In [ ]:
# ---- 10. DESCARGA DESDE GOOGLE DRIVE ----

drive_dl_results: list[BenchmarkResult] = []

# Ubicación real de los datos (ya sea recién subidos o existentes)
if isinstance(upload_result, dict) and upload_result.get('skipped'):
    DRIVE_SRC = upload_result['path']
else:
    DRIVE_SRC = DRIVE_MODEL_DIR

src_size = sum(f.stat().st_size for f in Path(DRIVE_SRC).rglob('*') if f.is_file())
print(f'  Source size: {src_size/1024**3:.2f} GB')

if not check_disk_space_needed(src_size):
    print('  ERROR: Disk space insufficient for Drive download benchmark')
else:
    def benchmark_drive_download(src, run_label):
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        dst = DOWNLOAD_BASE / f'drive_dl_{stamp}'
        dst.mkdir(parents=True)

        monitor = ResourceMonitor(str(dst))
        monitor.start()
        t0 = time.perf_counter()
        error = None
        try:
            shutil.copytree(src, str(dst), symlinks=False)
        except Exception as e:
            error = str(e)[:200]
            print(f'  ERROR: {error}')
        t1 = time.perf_counter()
        monitor.stop()

        total_time = t1 - t0
        actual_size = sum(f.stat().st_size for f in Path(dst).rglob('*') if f.is_file()) if not error else 0
        avg_speed = (actual_size / 1024**2) / total_time if total_time > 0 else 0
        mon_summary = monitor.get_summary()

        shutil.rmtree(str(dst), ignore_errors=True)

        return BenchmarkResult(
            model_id=model_id,
            method=f'drive_download_{run_label}',
            max_workers=0,
            total_size_bytes=actual_size,
            total_time_s=round(total_time, 1),
            avg_speed_mbps=round(avg_speed, 2),
            peak_speed_mbps=mon_summary.get('peak_download_speed_mbps', 0),
            file_count=sum(1 for _ in Path(dst).rglob('*') if _.is_file()) if not error else 0,
            avg_cpu_percent=mon_summary.get('avg_cpu_percent', 0),
            peak_cpu_percent=mon_summary.get('peak_cpu_percent', 0),
            avg_ram_percent=mon_summary.get('avg_ram_percent', 0),
            stall_count=mon_summary.get('stall_count', 0),
            stall_events=mon_summary.get('stall_events', []),
            error=error,
        )

    r1 = benchmark_drive_download(DRIVE_SRC, 'run1')
    drive_dl_results.append(r1)

    if not r1.error and r1.avg_speed_mbps > 75:
        n_extra = 2
        print(f'  Speed > 75 MB/s ({r1.avg_speed_mbps} MB/s) — repeating {n_extra} more runs...')
        for i in range(2, n_extra + 2):
            r = benchmark_drive_download(DRIVE_SRC, f'run{i}')
            drive_dl_results.append(r)
            time.sleep(2)

print('\nDrive download benchmark done')

## Resultados

In [ ]:
# ---- 11. TABLA DE RESULTADOS ----

all_results = hf_results + mw_results + ([upload_result] if not isinstance(upload_result, dict) else []) + drive_dl_results

def result_to_dict(r) -> dict:
    if isinstance(r, dict):
        return {'Modelo': model_id.split('/')[-1][:18], 'Metodo': 'drive_upload', 'Workers': 0,
                'Tamano (GB)': 0, 'Tiempo (s)': 0, 'Velocidad (MB/s)': 0,
                'Pico (MB/s)': 0, 'Archivos': 0, 'CPU avg (%)': 0, 'CPU peak (%)': 0,
                'RAM avg (%)': 0, 'Stalls': 0, 'Error': r.get('error', 'skipped')}
    return {
        'Modelo': r.model_id.split('/')[-1][:18],
        'Metodo': r.method,
        'Workers': r.max_workers,
        'Tamano (GB)': round(r.total_size_bytes / 1024**3, 2),
        'Tiempo (s)': r.total_time_s,
        'Velocidad (MB/s)': r.avg_speed_mbps,
        'Pico (MB/s)': r.peak_speed_mbps,
        'Archivos': r.file_count,
        'CPU avg (%)': r.avg_cpu_percent,
        'CPU peak (%)': r.peak_cpu_percent,
        'RAM avg (%)': r.avg_ram_percent,
        'Stalls': r.stall_count,
        'Error': r.error or '',
    }

df = pd.DataFrame([result_to_dict(r) for r in all_results])
display(df.style.background_gradient(subset=['Velocidad (MB/s)', 'Pico (MB/s)'], cmap='Greens'))

In [ ]:
# ---- 12. GRAFICAS ----

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'GenLab Benchmark: {model_id}', fontsize=14, fontweight='bold')

# 1. Velocidad HF normal vs xet_high
ax = axes[0]
plot_df = pd.DataFrame([result_to_dict(r) for r in hf_results if not r.error])
if not plot_df.empty:
    colors = {'normal': '#2196F3', 'xet_high': '#FF9800'}
    bars = ax.bar(plot_df['Metodo'], plot_df['Velocidad (MB/s)'],
                  color=[colors.get(m, '#999') for m in plot_df['Metodo']], alpha=0.85, width=0.5)
    for bar, val in zip(bars, plot_df['Velocidad (MB/s)']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('MB/s')
    ax.set_title('HF: normal vs xet_high')
    ax.grid(axis='y', alpha=0.3)

# 2. max_workers curve
ax = axes[1]
mw_plot = [r for r in mw_results if not r.error]
if len(mw_plot) > 1:
    workers = [r.max_workers for r in mw_plot]
    speeds = [r.avg_speed_mbps for r in mw_plot]
    ax.plot(workers, speeds, marker='o', color='#4CAF50', linewidth=2, markersize=8)
    ax.fill_between(workers, speeds, alpha=0.15, color='#4CAF50')
    ax.set_xlabel('max_workers')
    ax.set_ylabel('MB/s')
    ax.set_title('Velocidad vs max_workers')
    ax.set_xticks(workers)
    ax.grid(alpha=0.3)
    for w, s in zip(workers, speeds):
        ax.annotate(f'{s:.1f}', (w, s), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=8)
elif len(mw_plot) == 1:
    ax.text(0.5, 0.5, f'Single run: {mw_plot[0].avg_speed_mbps} MB/s', ha='center', va='center', transform=ax.transAxes)

# 3. Comparativa HF vs Drive
ax = axes[2]
compare_data = []
for r in hf_results:
    if not r.error:
        compare_data.append(('HF ' + r.method, r.avg_speed_mbps))
if isinstance(upload_result, BenchmarkResult) and not upload_result.error:
    compare_data.append(('Drive upload', upload_result.avg_speed_mbps))
for r in drive_dl_results:
    if not r.error:
        compare_data.append(('Drive dl ' + r.method.split('_')[-1], r.avg_speed_mbps))
if compare_data:
    labels, vals = zip(*compare_data)
    colors = ['#2196F3', '#FF9800', '#E91E63', '#9C27B0', '#009688'][:len(labels)]
    bars = ax.barh(range(len(labels)), vals, color=colors, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', ha='left', va='center', fontsize=9)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('MB/s')
    ax.set_title('Comparativa final')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- 13. STALLS REPORT ----

print('STALL REPORT (speed < 5 MB/s for > 10s)')
print('='*60)

stall_found = False
for r in all_results:
    if isinstance(r, BenchmarkResult) and r.stall_count > 0:
        stall_found = True
        print(f'  {r.method} [w={r.max_workers}]')
        for s in r.stall_events:
            print(f'     Stall: {s["duration_s"]}s at {s["speed_mbps"]} MB/s')

if not stall_found:
    print('  No stalls detected.')

print()
print('SUMMARY')
print('='*60)
if hf_results:
    for r in hf_results:
        if not r.error:
            print(f'  HF {r.method:10s}: {r.avg_speed_mbps:>6.1f} MB/s avg, '
                  f'CPU {r.avg_cpu_percent:>4.1f}%, RAM {r.avg_ram_percent:>4.1f}%, '
                  f'Stalls {r.stall_count}')
if isinstance(upload_result, BenchmarkResult) and not upload_result.error:
    r = upload_result
    print(f'  Drive up:    {r.avg_speed_mbps:>6.1f} MB/s avg, '
          f'CPU {r.avg_cpu_percent:>4.1f}%, RAM {r.avg_ram_percent:>4.1f}%, '
          f'Stalls {r.stall_count}')
if drive_dl_results:
    for r in drive_dl_results:
        if not r.error:
            print(f'  Drive dl {r.method.split("_")[-1]:4s}: {r.avg_speed_mbps:>6.1f} MB/s avg, '
                  f'CPU {r.avg_cpu_percent:>4.1f}%, RAM {r.avg_ram_percent:>4.1f}%, '
                  f'Stalls {r.stall_count}')

if hf_results and drive_dl_results:
    hf_avg = sum(r.avg_speed_mbps for r in hf_results if not r.error) / max(len([r for r in hf_results if not r.error]), 1)
    dd_avg = sum(r.avg_speed_mbps for r in drive_dl_results if not r.error) / max(len([r for r in drive_dl_results if not r.error]), 1)
    if dd_avg > hf_avg:
        print(f'\n  Drive download es {dd_avg/hf_avg:.1f}x más rápido que HF')
    else:
        print(f'\n  HF es {hf_avg/dd_avg:.1f}x más rápido que Drive download')

In [ ]:
# ---- 14. EXPORT ----

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

export_results = [r for r in all_results if isinstance(r, BenchmarkResult)]
df_export = pd.DataFrame([result_to_dict(r) for r in export_results])
for k, v in env.items():
    df_export[f'env_{k}'] = v

csv_path = f'/content/genlab_benchmark_{timestamp}.csv'
df_export.to_csv(csv_path, index=False)
print(f'CSV: {csv_path}')

detailed = []
for r in export_results:
    d = asdict(r)
    d['stall_events'] = [str(e) for e in d['stall_events']]
    detailed.append(d)

json_path = f'/content/genlab_benchmark_detailed_{timestamp}.json'
with open(json_path, 'w') as f:
    json.dump({'env': env, 'results': detailed, 'model_id': model_id}, f, indent=2, default=str)
print(f'JSON: {json_path}')

from google.colab import files
files.download(csv_path)
files.download(json_path)

# Copiar a Drive
import shutil
shutil.copy(csv_path, f'{DRIVE_BENCH_DIR}/')
shutil.copy(json_path, f'{DRIVE_BENCH_DIR}/')
print(f'Also saved to Drive: {DRIVE_BENCH_DIR}/')

In [ ]:
# ---- 15. CLEANUP ----

clear_hf_cache()
du = shutil.disk_usage('/')
print(f'Final free space: {du.free/1024**3:.1f} GB / {du.total/1024**3:.1f} GB')